In [1]:
%run start

Root set to: /home/bdudas/obesity_challange


In [2]:
import os
print("Training started")
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1,2,3"  
print("Using GPU:", os.environ["CUDA_VISIBLE_DEVICES"])
from src.data.vae_data import get_loaders
from src.models.transformerVAE import TransformerVAEEncoder, TransformerVAEDecoder,TransformerClassifier, Transfomer_latent_Classifier
from src.models.vae_trainers import StateTrainer, StateTrainer_latent
from src.utils import CosineMSELoss
import argparse
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import MLFlowLogger, WandbLogger
from omegaconf import OmegaConf


Training started
Using GPU: 0,1,2,3


/home/bdudas/anaconda3/envs/vcell/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
traning_config = OmegaConf.load("configs/traning.yaml")
run_name_base = traning_config.run_name
classtype = traning_config.classify_on


In [4]:
modelcpkt_adipo = ModelCheckpoint(monitor="Val/AUROC_adipo",save_top_k=3,mode="max")
modelcpkt_lipo = ModelCheckpoint(monitor="Val/AUROC_lipo",save_top_k=3,mode="max")
earlystop = pl.callbacks.EarlyStopping(monitor="Val/AUROC_adipo",patience=10,mode="max")
trainloader, valloader, *_ = get_loaders("",batch_size=traning_config.batch_size,num_workers = 10)   


In [5]:
encoder_config = OmegaConf.load("configs/encoder.yaml")
decoder_config = OmegaConf.load("configs/decoder.yaml")
if classtype == "latent":
    classifier_config = OmegaConf.load("configs/classifier_latent.yaml")
    print("Using latent classifier config")
else:
    classifier_config = OmegaConf.load("configs/classifier.yaml")
trainer_config = OmegaConf.load("configs/trainer.yaml")
z_dim = 256
encoder_config.z_dim = z_dim
decoder_config.z_dim = z_dim
classifier_config.z_dim = z_dim
trainer_config.classFactor = .9
traning_config.alpha = 0.1
trainer_config.kldFactor = 0.1


Using latent classifier config


In [6]:
encoder = TransformerVAEEncoder(**encoder_config)
decoder = TransformerVAEDecoder(**decoder_config)
loss = CosineMSELoss(alpha=traning_config.alpha)
classifier = TransformerClassifier(**classifier_config) if traning_config.classify_on =="after" else Transfomer_latent_Classifier(**classifier_config)
model = StateTrainer(encoder,decoder,categorizer=classifier,**trainer_config, reconLoss=loss) if traning_config.classify_on =="after" else StateTrainer_latent(encoder,decoder,categorizer=classifier,reconLoss=loss,**trainer_config)

mlfLogger = MLFlowLogger(experiment_name=traning_config.projectName,run_name = traning_config.run_name) 

/home/bdudas/anaconda3/envs/vcell/lib/python3.10/site-packages/torch/nn/modules/transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
/home/bdudas/anaconda3/envs/vcell/lib/python3.10/site-packages/pytorch_lightning/utilities/parsing.py:210: Attribute 'reconLoss' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['reconLoss'])`.
/home/bdudas/anaconda3/envs/vcell/lib/python3.10/site-packages/mlflow/tracking/_tracking_service/utils.py:177: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance. For migrating existing data, https://gith

In [7]:
mlfLogger = MLFlowLogger(experiment_name=traning_config.projectName,run_name = traning_config.run_name) 
trainer = pl.Trainer(max_epochs=traning_config.max_epochs, accelerator="auto", devices="auto",logger = mlfLogger, callbacks=[modelcpkt_adipo,modelcpkt_lipo,earlystop])

Trainer will use only 1 of 4 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=4)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


In [8]:
trainer.fit(model, train_dataloaders=trainloader, val_dataloaders=valloader)


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]

  | Name          | Type                         | Params | Mode  | FLOPs
-------------------------------------------------------------------------------
0 | encoder       | TransformerVAEEncoder        | 19.5 M | train | 0    
1 | decoder       | TransformerVAEDecoder        | 7.1 M  | train | 0    
2 | categorizer   | Transfomer_latent_Classifier | 1.6 M  | train | 0    
3 | aucMetric     | MulticlassAUROC              | 0      | train | 0    
4 | confmat       | MulticlassConfusionMatrix    | 0      | train | 0    
5 | classwise_auc | ClasswiseWrapper             | 0      | train | 0    
6 | EntropyLoss   | CrossEntropyLoss             | 0      | train | 0    
7 | reconLoss     | CosineMSELoss                | 0      | train | 0    
-------------------------------------------------------------------------------
28.2 M    Trainable params
0         Non-trainable params
28.2 M    Total params
112.926   Total estimated model params size 

Epoch 1:  78%|███████▊  | 431/552 [01:40<00:28,  4.29it/s, v_num=2f8f, Train/retrival_accuracy=0.00781, Train/loss_recon=2.080, Train/loss_kld=684.0, Train/beta=0.100, Train/loss_class=1.410, Train/total_loss=13.50, Val/retrival_accuracy=0.00573, Val/loss_recon=2.550, Val/loss_kld=477.0, Val/beta=0.100, Val/loss_class=1.350, Val/total_loss=13.90]


Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

/home/bdudas/anaconda3/envs/vcell/lib/python3.10/site-packages/IPython/core/interactiveshell.py:3587: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
